In [2]:
import geopandas as gpd
from shapely.ops import unary_union 
import unicodedata

In [6]:

c_2010 = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\soil_use_2010.shp")
c_2023 = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\soil_use_2023.shp")

In [14]:


# --- helpers rápidos ---
def normalize_txt(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return s.lower().strip()

def filtro_urbano(g: gpd.GeoDataFrame, coluna_classe="soil_use") -> gpd.GeoDataFrame:
    if coluna_classe not in g.columns:
        raise ValueError(f"Coluna de classe '{coluna_classe}' não encontrada")
    g = g.copy()
    g[coluna_classe] = g[coluna_classe].map(normalize_txt)
    return g[g[coluna_classe].str.contains(r"\burbano\b")].copy()

def fix_valid(g: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    try:
        from shapely import make_valid
        g["geometry"] = g.geometry.apply(make_valid)
    except Exception:
        g["geometry"] = g.buffer(0)
    return g

def to_same_crs(g1: gpd.GeoDataFrame, g2: gpd.GeoDataFrame) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    if g1.crs is None or g2.crs is None:
        raise ValueError("Algum layer está sem CRS; defina antes de continuar.")
    if g1.crs != g2.crs:
        g2 = g2.to_crs(g1.crs)
    return g1, g2

def diff_ano(g_new_u: gpd.GeoDataFrame, g_old_u: gpd.GeoDataFrame, ano_new: int, ano_old: int) -> gpd.GeoDataFrame:
    g_new_u = fix_valid(g_new_u)
    g_old_u = fix_valid(g_old_u)
    g_new_u, g_old_u = to_same_crs(g_new_u, g_old_u)

    new_u = unary_union(g_new_u.geometry)
    old_u = unary_union(g_old_u.geometry)
    d = new_u.difference(old_u)

    if d.is_empty:
        return gpd.GeoDataFrame(columns=["year_from", "year_to", "geometry"], crs=g_new_u.crs)

    gdiff = gpd.GeoDataFrame(geometry=[d], crs=g_new_u.crs).explode(index_parts=False, ignore_index=True)

    # (opcional) remover polígonos muito pequenos — reprojeta para métrico rápido
    try:
        g_m = gdiff.to_crs(3857)  # metros (aprox.)
        keep = g_m.area > 500     # ex.: > 500 m²; ajuste conforme necessário
        gdiff = gdiff[keep].copy()
    except Exception:
        pass

    # volta para WGS84 para o Folium (se quiser)
    if gdiff.crs.to_epsg() != 4326:
        gdiff = gdiff.to_crs(4326)

    gdiff["year_from"] = int(ano_old)
    gdiff["year_to"]   = int(ano_new)
    return gdiff

# --- uso ---
# Se você já tem gdf_2010 e gdf_2023 lidos:
gdf_2010 = c_2010
gdf_2023 = c_2023

g2010_u = filtro_urbano(gdf_2010, coluna_classe="soil_use")  # ajuste o nome da coluna se for outro
g2023_u = filtro_urbano(gdf_2023, coluna_classe="soil_use")

gdiff_10_23 = diff_ano(g_new_u=g2023_u, g_old_u=g2010_u, ano_new=2023, ano_old=2010)

print(gdiff_10_23.head())

Empty GeoDataFrame
Columns: [year_from, year_to, geometry]
Index: []


In [8]:
c_2010.head()

,class,soil_use,year,geometry
0,0,nao urbano,2010.0,"MULTIPOLYGON (((-46.86646 -23.09892, -46.86619..."
1,1,urbano,2010.0,"MULTIPOLYGON (((-46.87238 -23.08194, -46.87185..."


In [3]:
path = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\area_ponderacao_2023_com_mun\area_ponderacao_2023_com_mun.shp"

# ler o shapefile
gdf = gpd.read_file(path)

In [10]:
contagem = gdf.groupby("NM_MUN").size().reset_index(name="n_linhas")

In [11]:
contagem.sort_values("n_linhas", ascending=False)

,NM_MUN,n_linhas
572,São Paulo,310
100,Campinas,36
204,Guarulhos,30
340,Mogi das Cruzes,24
287,Jundiaí,24
...,...,...
15,Analândia,1
17,Angatuba,1
18,Anhembi,1
19,Anhumas,1


In [7]:


# Filtrar municípios com apenas 1 linha
municipios_1 = contagem[contagem["n_linhas"] == 1]

In [8]:
municipios_1

,CD_MUN,n_linhas
1,3500204,1
2,3500303,1
3,3500402,1
4,3500501,1
5,3500550,1
...,...,...
638,3556909,1
639,3556958,1
642,3557154,1
643,3557204,1


In [4]:
gdf

,CdAponOrNu,FZZ_URB_%,FZZ_RUR_%,FZZ_NAT_%,ID,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,CD_REGIAO,NM_REGIAO,CD_CONCURB,NM_CONCURB,AREA_KM2,geometry
0,3.505500e+12,90.924205,8.728386,0.347409,10000,3505500,Barretos,350032,Barretos,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,3505500,Barretos,1566.161,"POLYGON ((124120.018 7724241.606, 124124.31 77..."
1,3.505500e+12,65.678378,34.235772,0.085849,10001,3505500,Barretos,350032,Barretos,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,3505500,Barretos,1566.161,"POLYGON ((129263.274 7724764.877, 129266.423 7..."
2,3.505609e+12,94.412799,5.587201,0.000000,10002,3505609,Barrinha,350031,Ribeirão Preto,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,None,None,146.025,"POLYGON ((172702.323 7652971.977, 172258.466 7..."
3,3.505609e+12,42.725016,55.723274,1.551710,10003,3505609,Barrinha,350031,Ribeirão Preto,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,None,None,146.025,"POLYGON ((172136.189 7656728.11, 172160.682 76..."
4,3.505708e+12,99.203761,0.796239,0.000000,10004,3505708,Barueri,350001,São Paulo,3501,São Paulo,35,São Paulo,3,Sudeste,3550308,São Paulo/SP,65.701,"POLYGON ((308710.179 7400292.478, 308811.089 7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1874,3.505401e+12,26.780852,40.117500,33.101648,9995,3505401,Barra do Turvo,350005,Registro,3502,Sorocaba,35,São Paulo,3,Sudeste,None,None,1007.684,"POLYGON ((144223.544 7265166.769, 144679.283 7..."
1875,3.505500e+12,97.164482,2.835518,0.000000,9996,3505500,Barretos,350032,Barretos,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,3505500,Barretos,1566.161,"POLYGON ((128258.89 7726266.928, 128272.513 77..."
1876,3.505500e+12,36.460317,56.352938,7.186745,9997,3505500,Barretos,350032,Barretos,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,3505500,Barretos,1566.161,"POLYGON ((129541.829 7751214.758, 129641.368 7..."
1877,3.505500e+12,97.372720,2.627280,0.000000,9998,3505500,Barretos,350032,Barretos,3508,Ribeirão Preto,35,São Paulo,3,Sudeste,3505500,Barretos,1566.161,"POLYGON ((127158.206 7724179.744, 127163.243 7..."
